# LLM Prompt Runner

Run prompts on different LLM models (Colab AI, OpenAI). Experiments and prompts are defined in `start_here/experiments/definition.py` and stored in the database; the notebook loads them and writes results back to the DB.

In [ ]:
# (Optional) Install dependencies if needed (run once, e.g. in Colab)
# You can comment this out locally if packages are already installed.
#%pip install openai psycopg2-binary -q

In [ ]:
import sys
from pathlib import Path

# Dynamically locate the project root (directory that has both
# `integrations` and `managers`) starting from the current working dir
_cwd = Path.cwd().resolve()
_project_root = None
for candidate in [_cwd, *_cwd.parents]:
    if (candidate / "integrations").exists() and (candidate / "managers").exists():
        _project_root = candidate
        break

print("CWD:", _cwd)
print("Project root:", _project_root)

if _project_root is None:
    raise RuntimeError(
        f"Could not locate project root starting from {_cwd}. "
        "Make sure you run this notebook from inside the cloned repository."
    )

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
    print("Added to sys.path:", _project_root)

from datetime import datetime
from integrations.colab_llm_provider import ColabLLMProvider
from integrations.openai_llm_provider import OpenAILLMProvider
from managers.prompt_output_manager import save_results
from database.experiment_log import (
    list_experiments,
    get_experiment,
    get_prompts,
)
from start_here.experiments.constants import (
    MODEL_PROVIDERS,
    MODEL_KEYS,
    OPENAI_API_KEY,
)
import os
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY or "")

## Execution

### 0. Create tables and seed from experiments.py (run once)

Creates `experiment_setup`, `prompt`, and `llm_prompt_results` if they do not exist, then inserts experiment definitions from `start_here/experiments/definition.py`. Existing experiment names are skipped.

In [ ]:
from managers.database_manager import load_database
from importlib import reload

from start_here.experiments import constants
import database.experiment_log as experiment_log
import managers.database_manager as db_manager

reload(constants)
reload(experiment_log)
reload(db_manager)

print("DB name    :", constants.RESULTS_DATABASE_NAME)
print("DB host    :", constants.RESULTS_DATABASE_HOST)
print("DB port    :", constants.RESULTS_DATABASE_PORT)
print("DB user    :", constants.RESULTS_DATABASE_USER)
print("DB password:", constants.RESULTS_DATABASE_PASSWORD)

load_database()

### Available models

In [ ]:
print("Available models (use MODEL_KEYS[i] in step 1):")
for i, k in enumerate(MODEL_KEYS):
    print(f"  [{i}] {k} ({MODEL_PROVIDERS[k]})")

### 1. Choose or create experiment (model + decoding from DB)

In [ ]:
experiments = list_experiments()
for exp in experiments:
    print(f"  id={exp['id']} | {exp['name']} | {exp['model_key']} | t={exp['temperature']} top_p={exp['top_p']} max_tokens={exp['max_tokens']}")
if not experiments:
    print("  No experiments yet. Define them in start_here/experiments/definition.py and run load_database().")

In [ ]:
EXPERIMENT_ID = 3  # use an existing id from list_experiments()

experiment = get_experiment(EXPERIMENT_ID)
if experiment:
    print(f"Experiment: {experiment['name']} (id={EXPERIMENT_ID})")
    print(f"Model: {experiment['model_key']} | temperature={experiment['temperature']} | top_p={experiment['top_p']} | max_tokens={experiment['max_tokens']}")
else:
    print("No experiment with that id. Define experiments in start_here/experiments/definition.py and run load_database().")

### 3. Run prompt

In this step there are **two independent execution options**, and you should use **only one at a time** depending on your goal:

- **3.1 Run prompts from experiment definition**: runs the prompts loaded from the database for the current `EXPERIMENT_ID` (using the `prompt` table via `get_prompts`) and saves the results into `llm_prompt_results`.

- **3.2 Run code-review prompt for `mlcq_samples`**: iterates over rows in the `mlcq_samples` table, injects the `file_content` into the template `prompts/code-review-prompt.txt` (replacing `<code file>`), calls the LLM for each sample, and saves the results into `llm_prompt_results` with `dataset_id = mlcq_samples.id`.

Choose **either 3.1 or 3.2** based on the type of experiment you want to run, and execute only the corresponding code cell.

In [ ]:
def _get_llm(model_key: str):
    if model_key not in MODEL_PROVIDERS:
        raise ValueError(f"Model '{model_key}' not found. Available: {list(MODEL_PROVIDERS.keys())}")
    provider = MODEL_PROVIDERS[model_key]
    if provider == "colab":
        return ColabLLMProvider(model_key)
    if provider == "openai":
        return OpenAILLMProvider(model_key)
    raise ValueError(f"Unsupported provider: {provider}")


### 3.1 Run prompts from experiment definition

Uses the prompts, from the `prompt` table (via `get_prompts`), loaded for the `EXPERIMENT_ID` and saves the results to `llm_prompt_results`.

### 3.1.1 Load prompts for this experiment (from DB)

Run this cell after editing a prompt .txt file. It reloads prompts from disk into the DB for the current `EXPERIMENT_ID`.

In [ ]:
from managers.database_manager import refresh_prompts_from_file

refresh_prompts_from_file(EXPERIMENT_ID)
prompts_for_experiment = get_prompts(EXPERIMENT_ID)
PROMPTS = [p["prompt_text"] for p in prompts_for_experiment]
PROMPT_IDS = [p["id"] for p in prompts_for_experiment]
print(f"Loaded {len(PROMPTS)} prompt(s) for experiment {EXPERIMENT_ID}.")

In [ ]:
def _format_result(prompt, response, model_key, model_name, start_time, duration_seconds, success, prompt_id, error=None):
    return {
        "prompt": prompt,
        "response": response,
        "model_key": model_key,
        "model_name": model_name,
        "timestamp": start_time.isoformat(),
        "duration_seconds": duration_seconds,
        "success": success,
        "prompt_id": prompt_id,
        "error": error,
    }

def _execute_prompt(model_key: str, prompt: str, temperature: float, top_p: float, max_tokens: int, prompt_id: int):
    llm = _get_llm(model_key)
    start_time = datetime.now()
    try:
        response = llm.generate(prompt, temperature=temperature, top_p=top_p, max_tokens=max_tokens)
        duration = (datetime.now() - start_time).total_seconds()
        return _format_result(prompt, response, model_key, llm.model_name, start_time, duration, True, prompt_id)
    except Exception as e:
        duration = (datetime.now() - start_time).total_seconds()
        return _format_result(prompt, None, model_key, llm.model_name, start_time, duration, False, prompt_id, error=str(e))


results = []
for prompt_text, prompt_id in zip(PROMPTS, PROMPT_IDS):
    result = _execute_prompt(
        experiment["model_key"],
        prompt_text,
        experiment["temperature"],
        experiment["top_p"],
        experiment["max_tokens"],
        prompt_id,
    )
    results.append(result)

if results:
    save_results(EXPERIMENT_ID, results, experiment["model_key"], results[0]["model_name"])

### 3.2 Run code-review prompt for `mlcq_samples`

Uses `prompts/code-review-prompt.txt` and, for each selected row in `mlcq_samples`, replaces `<code file>` with `file_content`, calls the LLM, and stores the result in `llm_prompt_results` with `dataset_id = mlcq_samples.id`.

In [ ]:
from start_here.experiments.constants import (
    RESULTS_DATABASE_NAME,
    RESULTS_DATABASE_HOST,
    RESULTS_DATABASE_PORT,
    RESULTS_DATABASE_USER,
    RESULTS_DATABASE_PASSWORD,
)
from managers.prompt_input_manager import read_from_file
from database.experiment_log import add_prompt
import psycopg2

NUMBER_OF_SAMPLES_TO_REVIEW = 1  # samples per smell (adjust as needed)
SMELLS_TO_REVIEW = [
    "feature envy",
    "blob",
    "long method",
    "data class",
]  # adjust to the smells you want

base_prompt_text = read_from_file("code-review-prompt.txt")[0]

connection = psycopg2.connect(
    dbname=RESULTS_DATABASE_NAME,
    user=RESULTS_DATABASE_USER,
    password=RESULTS_DATABASE_PASSWORD,
    host=RESULTS_DATABASE_HOST,
    port=RESULTS_DATABASE_PORT,
)

try:
    all_samples = []
    with connection.cursor() as cursor:
        for smell in SMELLS_TO_REVIEW:
            cursor.execute(
                """
                SELECT id, file_content
                FROM mlcq_samples
                WHERE smell = %s
                AND severity = 'critical'
                ORDER BY id
                LIMIT %s
                """,
                (smell, NUMBER_OF_SAMPLES_TO_REVIEW),
            )
            samples_for_smell = cursor.fetchall()
            all_samples.extend(samples_for_smell)

    llm = _get_llm(experiment["model_key"])
    results = []

    for dataset_id, file_content in all_samples:
        if not file_content:
            continue

        prompt_text = base_prompt_text.replace("<code file>", file_content)
        prompt_id = add_prompt(EXPERIMENT_ID, prompt_text)

        start_time = datetime.now()
        try:
            response = llm.generate(
                prompt_text,
                temperature=experiment["temperature"],
                top_p=experiment["top_p"],
                max_tokens=experiment["max_tokens"],
            )
            duration = (datetime.now() - start_time).total_seconds()
            result = {
                "prompt": prompt_text,
                "response": response,
                "model_key": experiment["model_key"],
                "model_name": llm.model_name,
                "timestamp": start_time.isoformat(),
                "duration_seconds": duration,
                "success": True,
                "prompt_id": prompt_id,
                "dataset_id": dataset_id,
                "error": None,
            }
        except Exception as e:
            duration = (datetime.now() - start_time).total_seconds()
            result = {
                "prompt": prompt_text,
                "response": None,
                "model_key": experiment["model_key"],
                "model_name": llm.model_name,
                "timestamp": start_time.isoformat(),
                "duration_seconds": duration,
                "success": False,
                "prompt_id": prompt_id,
                "dataset_id": dataset_id,
                "error": str(e),
            }

        results.append(result)

    if results:
        save_results(EXPERIMENT_ID, results, experiment["model_key"], llm.model_name)
finally:
    connection.close()